In [1]:
from pathlib import Path
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

train_df = pd.read_csv(ROOT / "data" / "processed" / "train.csv")
val_df = pd.read_csv(ROOT / "data" / "processed" / "validation.csv")

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.0+cpu
CUDA available: False


In [3]:
X_train = train_df["clean_text"]
y_train = train_df["sentiment"]

X_val = val_df["clean_text"]
y_val = val_df["sentiment"]

classifiers = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),
    "Linear SVM": LinearSVC(class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )
}

results = []
trained_models = {}

for name, classifier in classifiers.items():

    model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                max_features=50000,
                min_df=2,
                sublinear_tf=True
            )
        ),
        ("classifier", classifier)
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, predictions),
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1": f1
    })

    trained_models[name] = model

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    "Macro F1",
    ascending=False
)

display(results_df)

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
1,Logistic Regression,0.7840,0.635108,0.669576,0.649445
2,Linear SVM,0.8025,0.638578,0.615033,0.624035
3,Random Forest,0.7865,0.668647,0.521888,0.536188
0,Naive Bayes,0.7215,0.573581,0.341061,0.294705


In [4]:
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]

model_dir = ROOT / "models" / "traditional_ml"
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    best_model,
    model_dir / "best_tfidf_model.joblib"
)

logistic_model = trained_models["Logistic Regression"]

joblib.dump(
    logistic_model,
    "../models/logistic_regression.pkl"
)

results_dir = ROOT / "outputs" / "evaluation"
results_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    results_dir / "traditional_validation_results.csv",
    index=False
)

print("Best validation model:", best_name)

Best validation model: Logistic Regression
